## Script to read in daily data from CAM (hopefully ERA5) and calculate lag covariance plot.

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import statsmodels.api as smti
import statsmodels.tsa as smts
import statsmodels.tsa.stattools as smt

import matplotlib.pyplot as mp

In [65]:
'''
    Calculate the lag (in time) correlation of a 1D array of 
'''


def correlation_calc(in_arr1,in_arr2,corr_nlag):

# Perform forwards and backwards correlation

    rin_arr1 = in_arr1.isel(time=slice(None, None, -1))
    rin_arr2 = in_arr2.isel(time=slice(None, None, -1))        
    
    backwards = smt.ccf(rin_arr1, rin_arr2,  adjusted=False,fft=True)[::-1]
    forwards = smt.ccf(in_arr1, in_arr2, adjusted=False,fft=True)

# Stitch together for continuous plot.

    corr_values = np.r_[backwards[:-1], forwards]
    
# Apply lag/lead values

    lstart = -backwards.size+1
    lstop =  backwards.size

    lags = np.arange(lstart,lstop)

    
    corr_values = xr.DataArray(corr_values,dims="lag",coords=dict(lag=lags), attrs  = {'units' : 'days'})      
   
    
    
# Note that both backwards and forwards contained lag 0, so we had to remove that from one of them when combining them.


    return corr_values

In [16]:
'''
    Remove seasonal cycle from whole time series
'''

def remove_seas_mean(da_in):
    # fit polynomial: x^2*b1 + x*b2 + ... + bn

    print(len(da_in))
    
    # Form array for repeating seasonal average.
    X = [i%365 for i in range(0, len(da_in))]
    print(len(X))
    
    degree = 5 # Polynomial order
    coef = np.polyfit(X, da_in.values, degree)
    print(len(coef))
    print('Coefficients: %s' % coef)
    
    # create curve
    curve = list()
    
    for i in range(len(X)):
        value = coef[-1]
        
# integrate order successively
        for d in range(degree):
            value += X[i]**(degree-d) * coef[d]
        curve.append(value)
    
        # plot curve over original data
    print(len(curve))
    mp.plot(da_in.values)
    mp.plot(curve, color='red', linewidth=3)
    mp.show()
    return da_in-curve


In [19]:
var1_name = 'PRECT'
var2_name = 'pr'
var3_name = 'PRECT'

#case = 'f.e20.FHIST.f09_f09.cesm2_1_capeten.001'
case1 = 'f.e20.FHIST.f09_f09.same_setting_143'

dir0 = '/glade/p/cgd/amp/rneale/'
#dir0 = '/Users/rneale/Documents/NCAR/CAM/'



dir_file = dir0+case1+'/tseries/'+case1+'_dmeans_ts_'+var1_name+'.nc'
ds1_ts = xr.open_dataset(dir_file,engine='netcdf4')#

#dir_file = dir0+case+'/tseries/'+case+'_dmeans_ts_'+var2_name+'.nc'
#ds2_ts = xr.open_dataset(dir_file,engine='netcdf4')


## Timeslice runs

#dir_tslice = '/glade/campaign/cgd/amp/bundy/mdtf/'
dir_tslice = '/glade/work/rneale/mdtf_timeslice/'
gfdl_dir = 'gfdl_timeslice'
ncar_dir = '' 



#file_lprec = 'atmos.1982010100-1982123123.prls.nc'

file_prec = 'atmos.1989010100-1989123123.pr.dmeans.nc'
dir_file = dir_tslice+'/am4/daily/'+file_prec

ds2_ts = xr.open_dataset(dir_file,engine='netcdf4')#

In [24]:
dir_obs = '/glade/work/rneale/data/TRMM/'
file_obs = dir_obs+'TRMM.PRECT.nc'

ds3_ts = xr.open_dataset(file_obs,engine='netcdf4')#



da1_ts = ds1_ts[var1_name]*86400.*1000.
da2_ts = ds2_ts[var2_name]*86400.*1000.
da3_ts = ds3_ts[var3_name]
da3_ts

<xarray.DataArray 'PRECT' (time: 4383, lat: 99, lon: 360)>
[156210120 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 1998-01-01 1998-01-02 ... 2009-12-31
  * lat      (lat) float64 -48.0 -47.02 -46.04 -45.06 ... 45.06 46.04 47.02 48.0
  * lon      (lon) float64 0.0 1.0 2.0 3.0 4.0 ... 355.0 356.0 357.0 358.0 359.0
Attributes:
    units:       mm/day
    long_name:   total daily precipitation
    delta_t:     0000-00-00 03:00:00
    avg_period:  0000-00-00 01:00:00
    remap:       remapped via ESMF_regrid_with_weights: Conservative remapping

In [69]:
#da_ts = da_ts.isel(time=slice(None, None, -1))

nlag = 30

#reg = 'SPCZ' ; min_lon = 180. ; max_lon = 190. ; min_lat = 10. ; max_lat = 15.
reg = 'East-Pacific' ; min_lon = 240. ; max_lon = 250. ; min_lat = 2.5 ; max_lat =7.5
#reg = 'West-Pacific' ; min_lon = 140. ; max_lon = 150. ; min_lat = 5. ; max_lat = 10.


mask_lon = (da1_ts.lon >= min_lon) & (da1_ts.lon <= max_lon)
mask_lat = (da1_ts.lat >= min_lat) & (da1_ts.lat <= max_lat)
da_x = da1_ts.where(mask_lon & mask_lat)

mask_lon = (da2_ts.lon >= min_lon) & (da2_ts.lon <= max_lon)
mask_lat = (da2_ts.lat >= min_lat) & (da2_ts.lat <= max_lat)
da_y = da2_ts.where(mask_lon & mask_lat, drop=True)

mask_lon = (da3_ts.lon >= min_lon) & (da3_ts.lon <= max_lon)
mask_lat = (da3_ts.lat >= min_lat) & (da3_ts.lat <= max_lat)
da_z = da3_ts.where(mask_lon & mask_lat, drop=True)

print(da_x)

<xarray.DataArray 'PRECT' (time: 12412, lat: 192, lon: 288)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan

In [75]:
#da_x = remove_seas_mean(da_x)
#da_y = remove_seas_mean(da_y)

acf = [[correlation_calc(da_x[:,jj,ii],da_x[:,jj,ii],nlag) for ii in da_x.lon] for jj in da_x.lat]
print(acf)

acf = correlation_calc(da_x,da_x,nlag)
acf.plot(xlim=(-nlag,nlag))
acf = correlation_calc(da_y,da_y,nlag)
acf.plot(xlim=(-nlag,nlag))
acf = correlation_calc(da_z,da_z,nlag)
acf.plot(xlim=(-nlag,nlag))

mp.xlabel('lag (days)')
mp.ylabel('correlation')
mp.axvline(x = 0, color = 'k',linestyle='dashed')
mp.axhline(y = 0, color = 'k',linestyle='dashed')
mp.legend(['CAM6','AM4','TRMM'])
mp.title(reg)
mp.savefig('autocorr.'+var1_name+'_'+reg+'.png', dpi=200,bbox_inches='tight') 

TypeError: invalid indexer array, does not have integer dtype: array(-90.)

In [ ]:
# Lag-correlation
#da_x.groupby([da_ts.index.day,da_x.index.month]).mean()
#da_ts.groupby("time.dayofyear").mean()




In [207]:
da_x = smt.add_constant(da_x)





In [209]:


#smt.acf(da_ts[50,50,:])[:20]
# performing the regression
# and fitting the model
result = smt.OLS(da_y, da_x,hasconst=False).fit()
 
# printing the summary table
dir(result)
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                      y   R-squared (uncentered):                   0.972
Model:                            OLS   Adj. R-squared (uncentered):              0.972
Method:                 Least Squares   F-statistic:                          2.158e+05
Date:                Tue, 24 Oct 2023   Prob (F-statistic):                        0.00
Time:                        16:43:40   Log-Likelihood:                         -63266.
No. Observations:               12410   AIC:                                  1.265e+05
Df Residuals:                   12408   BIC:                                  1.266e+05
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        240.6133      0.429    560.454      0.000     239.772     241.455
x1            -1.3779      0.046    -30.218      0.000      -1.467      -1.289
==============================================================================
Omnibus:                      614.033   Durbin-Watson:                   1.023
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              708.675
Skew:                          -0.584   Prob(JB):                    1.30e-154
Kurtosis:                       2.930   Cond. No.                         11.4
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""